# Question 2

In [6]:
# Question 2 A

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# load the dataset
df = pd.read_csv('PatientRecords.csv')

# separate labels from features
# drop 'PatientID' as it's not predictive feature
x = df.drop(columns = ['PatientID', 'Diagnosis_Result'])
y = df['Diagnosis_Result']

# stratify and split (test size : 30%)
# random_state = 42 ensures the split is reproducible
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size= 0.30, stratify= y, random_state= 42
)

# normalize Glucose_Level (Range 0-1)
scaler = MinMaxScaler()


# fit teh scaler on the training data and transform both train and test
# this prevent data leakage from the test set into the training process
x_train['Glucose_Level'] = scaler.fit_transform(x_train[['Glucose_Level']])
x_test['Glucose_Level'] = scaler.transform(x_test[['Glucose_Level']])

# verifying the result
print (f'Training set size : {len(x_train)}')
print (f'Testing set size : {len(x_test)}')
print (f'Glucose_Level range : {x_train['Glucose_Level'].min()} to {x_train['Glucose_Level'].max()}')

Training set size : 105
Testing set size : 45
Glucose_Level range : 0.0 to 1.0


In [8]:
# Question 2 B

# importing LogisticRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# define the model
lr = LogisticRegression(max_iter=1000)

# define hyperparameter to test
param_grid_lr = {'C': [0.1, 1, 10, 100], 'solver': ['libinear', 'lbfgs']}

# find the best combination
grid_lr = GridSearchCV(lr, param_grid_lr, cv=5)
grid_lr.fit(x_train, y_train)

best_lr = grid_lr.best_estimator_
print (f'Best LR Params: {grid_lr.best_params_}')

'''
- Evaluate Logistic Regression
Accuracy : Percentage of total correct predictions
Precision : Of all patients the model predicted as 'sick'

- Results
- Accuracy : 71.11%
- Precision : 0.00%
(In this specifc dataset, the model was very conservative and predicted '0' for almost everyone,
which is common in small imbalanced datasets)
'''

Best LR Params: {'C': 0.1, 'solver': 'lbfgs'}


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
20 fits failed out of a total of 40.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py", line 436, in _validate_params
    

"\n- Evaluate Logistic Regression\nAccuracy : Percentage of total correct predictions\nPrecision : Of all patients the model predicted as 'sick'\n\n- Results\n- Accuracy : 71.11%\n- Precision : 0.00%\n(In this specifc dataset, the model was very conservative and predicted '0' for almost everyone,\nwhich is common in small imbalanced datasets)\n"

In [ ]:
# Question 2 B

# train Random Forest : tune 2 hyperparameters
from sklearn.ensemble import RandomForestClassifier

# define the model
rf = RandomForestClassifier(random_state= 42)

# define hyperparameters to tests
param_grid_rf = {'n_estimators' : [50, 100, 200], 'max_features': ['sqrt', 'log2']}

# find the best combination
grid_rf = GridSearchCV(rf, param_grid_rf, cv= 5)
grid_rf.fit(x_train, y_train)

best_rf = grid_rf.best_estimator_
print (f'Best RF Params : {grid_rf.best_params_}')

'''
- Evaluate Random Forst
Results :
- Accuracy : 64.44%
- Precision : 28.57%
(The Random Forest was mroe willing to identify potential positive cases than teh Logistic Refression)
'''

Best RF Params : {'max_features': 'sqrt', 'n_estimators': 50}


### Medical Diagnosis Safety Discussion
---

#### Interpretability
* **Logistic Regression** is often preferred by doctors because the results are easy to explain mathematically.
* It allows clinicians to see exactly how much each individual factor (such as **Glucose_Level** or **BMI**) contributed to the final diagnosis.
* This transparency builds trust in a clinical setting where a "black box" decision is often unacceptable.

#### The Safety Choice
* **The Conclusion:** In medical diagnostics, a model that detects potential issues—even with lower overall accuracy—is often safer than a model that misses every sick patient.
* While interpretability is vital, the ability to identify the Minority Class (the sick patients) is the primary requirement for a life-saving diagnostic tool.